# Sesión 7 — Capa Gold, KPIs y Dashboards

**Objetivo:** construir la primera capa Gold formal del curso desde Silver, publicar KPIs comerciales de Lumi y KPIs operativos de Bagazo, y preparar datasets para dashboards.

## Reglas de seguridad

- Solo se lee desde `workspace.lumi_silver`, `workspace.bagazo_silver` y `workspace.control`.
- Solo se escribe en `workspace.lumi_gold`, `workspace.bagazo_gold` y `workspace.control`.
- `workspace.delta_lab` se mantiene como evidencia pedagógica de la Sesión 6, pero **no alimenta Gold**.
- No ejecutar `VACUUM`, `RESTORE`, `UPDATE`, `DELETE` ni `MERGE` sobre Silver.

> Nota importante: este notebook fue ajustado para usar los nombres reales observados en Silver, por ejemplo `order_purchase_ts`, `order_purchase_date`, `order_approved_ts` y `order_delivered_customer_ts`. Además, incluye helpers de PySpark para evitar errores por pequeñas diferencias de nombres de columnas.


In [0]:
%sql
USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS workspace.lumi_gold;
CREATE SCHEMA IF NOT EXISTS workspace.bagazo_gold;
CREATE SCHEMA IF NOT EXISTS workspace.control;


## 1. Retomar calidad y confiabilidad de sesiones anteriores

Antes de publicar Gold, revisamos dos evidencias:

1. Calidad de Silver creada en la Sesión 4.
2. Confiabilidad Delta revisada en la Sesión 6.

`reviews_clean` continúa en estado **REVISAR**, por eso sus métricas se agregan a nivel `order_id` antes de cruzarlas con órdenes, ítems o categorías.


In [0]:
%sql
SELECT *
FROM workspace.control.quality_summary_sesion_04
ORDER BY dataset, tabla;


In [0]:
%sql
SELECT *
FROM workspace.control.delta_reliability_summary_sesion_06
ORDER BY tabla;


## 2. Inventario de fuentes oficiales

Gold parte de Silver. Las tablas de `workspace.delta_lab` no se usan como fuente de negocio porque fueron tablas de laboratorio de la Sesión 6.


In [0]:
%sql
SHOW TABLES IN workspace.lumi_silver;


In [0]:
%sql
SHOW TABLES IN workspace.bagazo_silver;


## 3. Helpers robustos para construir Gold

Las tablas Silver pueden tener nombres ya estandarizados, por ejemplo `order_purchase_ts` en lugar de `order_purchase_timestamp`. Para evitar errores de columnas no resueltas, esta sección detecta columnas disponibles y selecciona la primera columna válida de una lista de candidatos.


In [0]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import DataFrame

CATALOGO = "workspace"
LUMI_SILVER = "workspace.lumi_silver"
BAGAZO_SILVER = "workspace.bagazo_silver"
LUMI_GOLD = "workspace.lumi_gold"
BAGAZO_GOLD = "workspace.bagazo_gold"
CONTROL = "workspace.control"

spark.sql(f"USE CATALOG {CATALOGO}")


def qcol(nombre: str) -> str:
    """Escapa nombres de columnas para expresiones SQL."""
    return "`" + nombre.replace("`", "``") + "`"


def existing_col(df: DataFrame, candidates):
    """Devuelve el primer nombre de columna existente dentro de una lista de candidatos."""
    for c in candidates:
        if c in df.columns:
            return c
    return None


def pick(df: DataFrame, candidates, alias: str, default=None, data_type: str | None = None):
    """Selecciona una columna existente o un valor por defecto, con cast tolerante cuando aplica."""
    c = existing_col(df, candidates)
    if c is None:
        expr = F.lit(default)
        if data_type:
            expr = expr.cast(data_type)
        return expr.alias(alias)
    if data_type and data_type.upper() not in ["STRING"]:
        expr = F.expr(f"try_cast({qcol(c)} AS {data_type})")
    elif data_type:
        expr = F.col(c).cast(data_type)
    else:
        expr = F.col(c)
    return expr.alias(alias)


def coalesce_pick(df: DataFrame, candidates, alias: str, data_type: str, default=None):
    """Hace coalesce entre todas las columnas candidatas existentes."""
    exprs = []
    for c in candidates:
        if c in df.columns:
            if data_type.upper() == "DATE":
                exprs.append(F.to_date(F.col(c)))
            elif data_type.upper() == "TIMESTAMP":
                exprs.append(F.to_timestamp(F.col(c)))
            elif data_type.upper() == "STRING":
                exprs.append(F.col(c).cast("string"))
            else:
                exprs.append(F.expr(f"try_cast({qcol(c)} AS {data_type})"))
    if not exprs:
        return F.lit(default).cast(data_type).alias(alias)
    return F.coalesce(*exprs).alias(alias)


def write_table(df: DataFrame, full_name: str):
    """Escribe una tabla Delta administrada, reemplazando solo tablas Gold/Control."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
    print(f"OK -> {full_name}: {spark.table(full_name).count():,} filas")


def preview_schema(table_name: str):
    df = spark.table(table_name)
    return spark.createDataFrame([(table_name, c, t) for c, t in df.dtypes], ["tabla", "columna", "tipo"])


## 4. Diagnóstico rápido de esquemas Silver

Este bloque ayuda a confirmar los nombres reales de columnas. No modifica nada.


In [0]:
schemas_df = (
    preview_schema("workspace.lumi_silver.orders_clean")
    .unionByName(preview_schema("workspace.lumi_silver.order_items_clean"))
    .unionByName(preview_schema("workspace.lumi_silver.payments_clean"))
    .unionByName(preview_schema("workspace.lumi_silver.reviews_clean"))
    .unionByName(preview_schema("workspace.bagazo_silver.operacion_ingenios_clean"))
)
display(schemas_df)


## 5. Gold Lumi — dimensiones

Se crean dimensiones simples y explicables: producto, cliente, vendedor y fecha.


In [0]:
products = spark.table("workspace.lumi_silver.products_clean")
customers = spark.table("workspace.lumi_silver.customers_clean")
sellers = spark.table("workspace.lumi_silver.sellers_clean")
orders = spark.table("workspace.lumi_silver.orders_clean")

# Dimensión producto
categoria = F.coalesce(
    *[F.col(c).cast("string") for c in [
        "categoria_producto",
        "product_category_name_english",
        "category_name_english",
        "product_category_name"
    ] if c in products.columns],
    F.lit("sin_categoria")
) if any(c in products.columns for c in ["categoria_producto", "product_category_name_english", "category_name_english", "product_category_name"]) else F.lit("sin_categoria")

dim_product = products.select(
    pick(products, ["product_id"], "product_id", data_type="STRING"),
    categoria.alias("categoria_producto"),
    pick(products, ["product_category_name"], "product_category_name", data_type="STRING"),
    pick(products, ["product_category_name_english", "category_name_english", "categoria_producto"], "product_category_name_english", data_type="STRING"),
    pick(products, ["product_weight_g"], "product_weight_g", data_type="DOUBLE"),
    pick(products, ["product_length_cm"], "product_length_cm", data_type="DOUBLE"),
    pick(products, ["product_height_cm"], "product_height_cm", data_type="DOUBLE"),
    pick(products, ["product_width_cm"], "product_width_cm", data_type="DOUBLE")
).withColumn(
    "categoria_requiere_revision",
    F.when(
        F.col("categoria_producto").isNull() |
        (F.lower(F.col("categoria_producto")).isin("sin_categoria", "not_defined", "sin_definir")),
        F.lit(True)
    ).otherwise(F.lit(False))
)
write_table(dim_product, "workspace.lumi_gold.dim_product")

# Dimensión cliente
dim_customer = customers.select(
    pick(customers, ["customer_id"], "customer_id", data_type="STRING"),
    pick(customers, ["customer_unique_id"], "customer_unique_id", data_type="STRING"),
    pick(customers, ["customer_zip_code_prefix"], "customer_zip_code_prefix", data_type="STRING"),
    pick(customers, ["customer_city"], "customer_city", data_type="STRING"),
    pick(customers, ["customer_state"], "customer_state", data_type="STRING")
)
write_table(dim_customer, "workspace.lumi_gold.dim_customer")

# Dimensión vendedor
dim_seller = sellers.select(
    pick(sellers, ["seller_id"], "seller_id", data_type="STRING"),
    pick(sellers, ["seller_zip_code_prefix"], "seller_zip_code_prefix", data_type="STRING"),
    pick(sellers, ["seller_city"], "seller_city", data_type="STRING"),
    pick(sellers, ["seller_state"], "seller_state", data_type="STRING")
)
write_table(dim_seller, "workspace.lumi_gold.dim_seller")

# Dimensión fecha construida desde las fechas reales disponibles en orders_clean
fecha_groups = [
    ["order_purchase_date", "order_purchase_ts", "order_purchase_timestamp"],
    ["order_approved_date", "order_approved_ts", "order_approved_at"],
    ["order_delivered_customer_date", "order_delivered_customer_ts"],
    ["order_estimated_delivery_date", "order_estimated_delivery_ts"]
]
fecha_dfs = []
for group in fecha_groups:
    if existing_col(orders, group):
        fecha_dfs.append(
            orders.select(coalesce_pick(orders, group, "fecha", "DATE"))
            .where(F.col("fecha").isNotNull())
        )

if fecha_dfs:
    fechas = reduce(lambda a, b: a.unionByName(b), fecha_dfs).distinct()
else:
    fechas = spark.createDataFrame([], T.StructType([T.StructField("fecha", T.DateType(), True)]))

dim_date = fechas.select(
    F.col("fecha"),
    F.year("fecha").alias("anio"),
    F.month("fecha").alias("mes"),
    F.date_format("fecha", "yyyy-MM").alias("anio_mes"),
    F.dayofmonth("fecha").alias("dia_mes"),
    F.dayofweek("fecha").alias("dia_semana"),
    F.quarter("fecha").alias("trimestre")
)
write_table(dim_date, "workspace.lumi_gold.dim_date")


## 6. Gold Lumi — hechos

Granularidades de referencia:

- `fact_orders`: una fila por `order_id`.
- `fact_sales_items`: una fila por `order_id + order_item_id`.
- `fact_payments`: una fila por `order_id`.
- `fact_delivery_experience`: una fila por `order_id`, con reviews agregadas previamente.


In [0]:
orders = spark.table("workspace.lumi_silver.orders_clean")
customers = spark.table("workspace.lumi_silver.customers_clean")
items = spark.table("workspace.lumi_silver.order_items_clean")
payments = spark.table("workspace.lumi_silver.payments_clean")
reviews = spark.table("workspace.lumi_silver.reviews_clean")
dim_product = spark.table("workspace.lumi_gold.dim_product")

# -----------------------------
# fact_orders
# -----------------------------
purchase_date_expr = coalesce_pick(orders, ["order_purchase_date", "order_purchase_ts", "order_purchase_timestamp"], "purchase_date", "DATE")
purchase_ts_expr = coalesce_pick(orders, ["order_purchase_ts", "order_purchase_timestamp", "order_purchase_date"], "order_purchase_ts", "TIMESTAMP")
approved_ts_expr = coalesce_pick(orders, ["order_approved_ts", "order_approved_at", "order_approved_date"], "order_approved_ts", "TIMESTAMP")
delivered_date_expr = coalesce_pick(orders, ["order_delivered_customer_date", "order_delivered_customer_ts"], "delivered_date", "DATE")
delivered_ts_expr = coalesce_pick(orders, ["order_delivered_customer_ts", "order_delivered_customer_date"], "order_delivered_customer_ts", "TIMESTAMP")
estimated_date_expr = coalesce_pick(orders, ["order_estimated_delivery_date", "order_estimated_delivery_ts"], "estimated_delivery_date", "DATE")

orders_base = orders.select(
    pick(orders, ["order_id"], "order_id", data_type="STRING"),
    pick(orders, ["customer_id"], "customer_id", data_type="STRING"),
    pick(orders, ["order_status"], "order_status", data_type="STRING"),
    purchase_date_expr,
    purchase_ts_expr,
    approved_ts_expr,
    delivered_date_expr,
    delivered_ts_expr,
    estimated_date_expr,
    pick(orders, ["delivery_days"], "delivery_days_original", data_type="INT"),
    pick(orders, ["delay_days"], "delay_days_original", data_type="INT"),
    pick(orders, ["is_late"], "is_late_original", data_type="BOOLEAN")
).withColumn(
    "purchase_month", F.date_format("purchase_date", "yyyy-MM")
).withColumn(
    "delivery_days",
    F.coalesce(F.col("delivery_days_original"), F.datediff(F.col("delivered_date"), F.col("purchase_date")))
).withColumn(
    "delay_days",
    F.coalesce(F.col("delay_days_original"), F.datediff(F.col("delivered_date"), F.col("estimated_delivery_date")))
).withColumn(
    "is_late",
    F.coalesce(F.col("is_late_original"), F.when(F.col("delay_days").isNull(), F.lit(False)).otherwise(F.col("delay_days") > 0))
).drop("delivery_days_original", "delay_days_original", "is_late_original")

customers_sel = customers.select(
    pick(customers, ["customer_id"], "customer_id", data_type="STRING"),
    pick(customers, ["customer_unique_id"], "customer_unique_id", data_type="STRING"),
    pick(customers, ["customer_state"], "customer_state", data_type="STRING"),
    pick(customers, ["customer_city"], "customer_city", data_type="STRING")
)

fact_orders = orders_base.join(customers_sel, on="customer_id", how="left")
write_table(fact_orders, "workspace.lumi_gold.fact_orders")

# -----------------------------
# fact_sales_items
# -----------------------------
items_base = items.select(
    pick(items, ["order_id"], "order_id", data_type="STRING"),
    pick(items, ["order_item_id"], "order_item_id", data_type="INT"),
    pick(items, ["product_id"], "product_id", data_type="STRING"),
    pick(items, ["seller_id"], "seller_id", data_type="STRING"),
    coalesce_pick(items, ["shipping_limit_date", "shipping_limit_ts"], "shipping_limit_date", "DATE"),
    pick(items, ["price", "item_price"], "item_price", default=0.0, data_type="DOUBLE"),
    pick(items, ["freight_value", "valor_flete"], "freight_value", default=0.0, data_type="DOUBLE")
).withColumn(
    "item_total_value", F.coalesce(F.col("item_price"), F.lit(0.0)) + F.coalesce(F.col("freight_value"), F.lit(0.0))
)

fact_sales_items = items_base.join(
    dim_product.select("product_id", "categoria_producto"),
    on="product_id",
    how="left"
).withColumn("categoria_producto", F.coalesce(F.col("categoria_producto"), F.lit("sin_categoria")))
write_table(fact_sales_items, "workspace.lumi_gold.fact_sales_items")

# -----------------------------
# fact_payments agregado a nivel order_id
# -----------------------------
payments_base = payments.select(
    pick(payments, ["order_id"], "order_id", data_type="STRING"),
    pick(payments, ["payment_type", "main_payment_type"], "payment_type", default="sin_dato", data_type="STRING"),
    pick(payments, ["payment_installments", "total_installments"], "payment_installments", default=0, data_type="INT"),
    pick(payments, ["payment_value", "payment_value_total", "valor_pagado_total"], "payment_value", default=0.0, data_type="DOUBLE")
)

fact_payments = payments_base.groupBy("order_id").agg(
    F.count(F.lit(1)).alias("payment_records"),
    F.countDistinct("payment_type").alias("payment_type_count"),
    F.max("payment_type").alias("main_payment_type"),
    F.sum("payment_installments").alias("total_installments"),
    F.sum("payment_value").alias("payment_value_total")
)
write_table(fact_payments, "workspace.lumi_gold.fact_payments")

# -----------------------------
# fact_delivery_experience agregado a nivel order_id
# -----------------------------
comment_expr = F.coalesce(
    *[F.col(c).cast("string") for c in ["review_comment_message", "review_comment_title", "comment", "comentario"] if c in reviews.columns],
    F.lit(None).cast("string")
) if any(c in reviews.columns for c in ["review_comment_message", "review_comment_title", "comment", "comentario"]) else F.lit(None).cast("string")

reviews_base = reviews.select(
    pick(reviews, ["order_id"], "order_id", data_type="STRING"),
    pick(reviews, ["review_score"], "review_score", data_type="DOUBLE"),
    comment_expr.alias("review_comment_text")
)

reviews_por_orden = reviews_base.groupBy("order_id").agg(
    F.count(F.lit(1)).alias("review_records"),
    F.avg("review_score").alias("review_score_avg"),
    F.min("review_score").cast("int").alias("review_score_min"),
    F.max("review_score").cast("int").alias("review_score_max"),
    F.sum(F.when(F.col("review_score") <= 2, 1).otherwise(0)).alias("low_review_records"),
    F.max(F.when(F.length(F.trim(F.col("review_comment_text"))) > 0, 1).otherwise(0)).alias("has_review_comment")
)

fact_delivery_experience = spark.table("workspace.lumi_gold.fact_orders").join(
    reviews_por_orden,
    on="order_id",
    how="left"
).select(
    "order_id", "customer_id", "order_status", "purchase_date", "purchase_month", "delivery_days", "delay_days", "is_late",
    F.coalesce(F.col("review_records"), F.lit(0)).alias("review_records"),
    "review_score_avg", "review_score_min", "review_score_max",
    F.coalesce(F.col("low_review_records"), F.lit(0)).alias("low_review_records"),
    F.coalesce(F.col("has_review_comment"), F.lit(0)).alias("has_review_comment")
).withColumn(
    "tramo_demora",
    F.when(F.col("delay_days").isNull(), "sin_dato")
     .when(F.col("delay_days") <= 0, "a_tiempo")
     .when(F.col("delay_days").between(1, 7), "tarde_1_7_dias")
     .when(F.col("delay_days").between(8, 15), "tarde_8_15_dias")
     .otherwise("tarde_mas_15_dias")
)
write_table(fact_delivery_experience, "workspace.lumi_gold.fact_delivery_experience")


## 7. KPIs Lumi

Ahora convertimos hechos en tablas de KPI listas para dashboard. Todas estas tablas parten de Gold, no directamente de Silver.


In [0]:
fo = spark.table("workspace.lumi_gold.fact_orders")
fi = spark.table("workspace.lumi_gold.fact_sales_items")
fp = spark.table("workspace.lumi_gold.fact_payments")
fe = spark.table("workspace.lumi_gold.fact_delivery_experience")

# Ventas mensuales
monthly_base = fo.join(fi, on="order_id", how="left").where(F.col("purchase_month").isNotNull())
kpi_monthly_sales = monthly_base.groupBy("purchase_month").agg(
    F.countDistinct("order_id").alias("pedidos"),
    F.sum("item_price").alias("venta_items"),
    F.sum("freight_value").alias("valor_flete"),
    F.sum("item_total_value").alias("venta_total_items"),
    F.avg(F.col("is_late").cast("int")).alias("tasa_entrega_tardia")
).withColumn(
    "ticket_promedio_items",
    F.round(F.col("venta_total_items") / F.when(F.col("pedidos") == 0, None).otherwise(F.col("pedidos")), 2)
).withColumn("tasa_entrega_tardia", F.round("tasa_entrega_tardia", 4))
write_table(kpi_monthly_sales, "workspace.lumi_gold.kpi_monthly_sales")

# Desempeño por categoría
category_base = fi.join(fe.select("order_id", "review_score_avg", "is_late"), on="order_id", how="left")
kpi_category_performance = category_base.groupBy("categoria_producto").agg(
    F.countDistinct("order_id").alias("pedidos"),
    F.count(F.lit(1)).alias("items_vendidos"),
    F.sum("item_total_value").alias("venta_total_items"),
    F.round(F.avg("review_score_avg"), 2).alias("review_promedio"),
    F.round(F.avg(F.col("is_late").cast("int")), 4).alias("tasa_entrega_tardia")
)
write_table(kpi_category_performance, "workspace.lumi_gold.kpi_category_performance")

# Métodos de pago
kpi_payment_methods = fp.groupBy("main_payment_type").agg(
    F.countDistinct("order_id").alias("pedidos"),
    F.sum("payment_value_total").alias("valor_pagado_total"),
    F.round(F.avg("payment_value_total"), 2).alias("pago_promedio_por_pedido"),
    F.round(F.avg("total_installments"), 2).alias("cuotas_promedio")
).withColumnRenamed("main_payment_type", "payment_type")
write_table(kpi_payment_methods, "workspace.lumi_gold.kpi_payment_methods")

# Demora vs review
kpi_delivery_review = fe.groupBy("tramo_demora").agg(
    F.countDistinct("order_id").alias("pedidos"),
    F.round(F.avg("review_score_avg"), 2).alias("review_promedio"),
    F.round(F.avg("delay_days"), 2).alias("demora_promedio_dias"),
    F.round(F.avg(F.col("is_late").cast("int")), 4).alias("tasa_entrega_tardia"),
    F.sum("low_review_records").alias("reviews_bajas")
)
write_table(kpi_delivery_review, "workspace.lumi_gold.kpi_delivery_review")

# Experiencia por estado del cliente
kpi_customer_experience = fo.join(fe.select("order_id", "review_score_avg", "low_review_records"), on="order_id", how="left") \
    .groupBy("customer_state").agg(
        F.countDistinct("order_id").alias("pedidos"),
        F.round(F.avg("review_score_avg"), 2).alias("review_promedio"),
        F.round(F.avg(F.col("is_late").cast("int")), 4).alias("tasa_entrega_tardia"),
        F.sum(F.coalesce(F.col("low_review_records"), F.lit(0))).alias("reviews_bajas")
    )
write_table(kpi_customer_experience, "workspace.lumi_gold.kpi_customer_experience")


### Validación rápida Lumi

Esta validación confirma que las tablas a nivel pedido conservan una fila por `order_id`.


In [0]:
%sql
SELECT 'fact_orders' AS tabla, COUNT(*) AS filas, COUNT(DISTINCT order_id) AS ordenes_distintas FROM workspace.lumi_gold.fact_orders
UNION ALL
SELECT 'fact_payments' AS tabla, COUNT(*) AS filas, COUNT(DISTINCT order_id) AS ordenes_distintas FROM workspace.lumi_gold.fact_payments
UNION ALL
SELECT 'fact_delivery_experience' AS tabla, COUNT(*) AS filas, COUNT(DISTINCT order_id) AS ordenes_distintas FROM workspace.lumi_gold.fact_delivery_experience;


## 8. Gold Bagazo — fact y KPIs

Bagazo mantiene granularidad `fecha + ingenio`. Esta tabla queda lista para KPIs y como base para feature engineering en la Sesión 8.


In [0]:
bagazo = spark.table("workspace.bagazo_silver.operacion_ingenios_clean")

bagazo_base = bagazo.select(
    coalesce_pick(bagazo, ["fecha"], "fecha", "DATE"),
    pick(bagazo, ["ingenio"], "ingenio", data_type="STRING"),
    pick(bagazo, ["anio"], "anio_original", data_type="INT"),
    pick(bagazo, ["mes"], "mes_original", data_type="INT"),
    pick(bagazo, ["dia_semana"], "dia_semana_original", data_type="INT"),
    pick(bagazo, ["lluvia_mm", "promedio_lluvias_mm"], "lluvia_mm", data_type="DOUBLE"),
    pick(bagazo, ["cana_molida_ton", "cana_molida_toneladas"], "cana_molida_ton", data_type="DOUBLE"),
    pick(bagazo, ["bagazo_entregado_ton", "bagazo_entregado_toneladas"], "bagazo_entregado_ton", data_type="DOUBLE"),
    pick(bagazo, ["comentario", "comentarios", "comentario_operativo"], "comentario", data_type="STRING"),
    pick(bagazo, ["lluvia_alta"], "lluvia_alta_original", data_type="BOOLEAN"),
    pick(bagazo, ["riesgo_bajo_bagazo"], "riesgo_bajo_bagazo_original", data_type="BOOLEAN"),
    pick(bagazo, ["tiene_comentario_operativo"], "tiene_comentario_operativo_original", data_type="BOOLEAN"),
    pick(bagazo, ["es_temporada_lluviosa", "temporada_lluviosa"], "es_temporada_lluviosa_original", data_type="BOOLEAN")
).withColumn(
    "anio", F.coalesce(F.col("anio_original"), F.year("fecha"))
).withColumn(
    "mes", F.coalesce(F.col("mes_original"), F.month("fecha"))
).withColumn(
    "dia_semana", F.coalesce(F.col("dia_semana_original"), F.dayofweek("fecha"))
).drop("anio_original", "mes_original", "dia_semana_original")

umbrales = bagazo_base.groupBy("ingenio").agg(
    F.expr("percentile_approx(bagazo_entregado_ton, 0.25)").alias("bagazo_p25_ingenio")
)

fact_operacion_ingenios = bagazo_base.join(umbrales, on="ingenio", how="left").withColumn(
    "lluvia_alta",
    F.coalesce(F.col("lluvia_alta_original"), F.when(F.col("lluvia_mm") >= 30, True).otherwise(False))
).withColumn(
    "riesgo_bajo_bagazo",
    F.coalesce(
        F.col("riesgo_bajo_bagazo_original"),
        F.when(F.col("bagazo_entregado_ton").isNull() | F.col("bagazo_p25_ingenio").isNull(), False)
         .otherwise(F.col("bagazo_entregado_ton") <= F.col("bagazo_p25_ingenio"))
    )
).withColumn(
    "tiene_comentario_operativo",
    F.coalesce(
        F.col("tiene_comentario_operativo_original"),
        F.when(F.length(F.trim(F.col("comentario"))) > 0, True).otherwise(False)
    )
).withColumn(
    "es_temporada_lluviosa",
    F.coalesce(F.col("es_temporada_lluviosa_original"), F.col("mes").isin(4, 5, 10, 11))
).withColumn(
    "anio_mes", F.date_format("fecha", "yyyy-MM")
).withColumn(
    "tramo_lluvia",
    F.when(F.col("lluvia_mm").isNull(), "sin_dato_lluvia")
     .when(F.col("lluvia_mm") == 0, "seco")
     .when((F.col("lluvia_mm") > 0) & (F.col("lluvia_mm") < 10), "lluvia_baja")
     .when((F.col("lluvia_mm") >= 10) & (F.col("lluvia_mm") < 30), "lluvia_media")
     .otherwise("lluvia_alta")
).drop(
    "lluvia_alta_original", "riesgo_bajo_bagazo_original", "tiene_comentario_operativo_original",
    "es_temporada_lluviosa_original", "bagazo_p25_ingenio"
)

write_table(fact_operacion_ingenios, "workspace.bagazo_gold.fact_operacion_ingenios")

fb = spark.table("workspace.bagazo_gold.fact_operacion_ingenios")

kpi_lluvia_bagazo_mensual = fb.groupBy("anio_mes", "ingenio").agg(
    F.count(F.lit(1)).alias("dias_observados"),
    F.round(F.avg("lluvia_mm"), 2).alias("lluvia_promedio_mm"),
    F.round(F.sum("lluvia_mm"), 2).alias("lluvia_total_mm"),
    F.round(F.avg("cana_molida_ton"), 2).alias("cana_promedio_ton"),
    F.round(F.avg("bagazo_entregado_ton"), 2).alias("bagazo_promedio_ton"),
    F.round(F.sum("bagazo_entregado_ton"), 2).alias("bagazo_total_ton"),
    F.sum(F.col("riesgo_bajo_bagazo").cast("int")).alias("dias_riesgo_bajo_bagazo")
)
write_table(kpi_lluvia_bagazo_mensual, "workspace.bagazo_gold.kpi_lluvia_bagazo_mensual")

kpi_bagazo_por_ingenio = fb.groupBy("ingenio").agg(
    F.count(F.lit(1)).alias("dias_observados"),
    F.round(F.avg("bagazo_entregado_ton"), 2).alias("bagazo_promedio_ton"),
    F.round(F.sum("bagazo_entregado_ton"), 2).alias("bagazo_total_ton"),
    F.round(F.avg("cana_molida_ton"), 2).alias("cana_promedio_ton"),
    F.round(F.avg("lluvia_mm"), 2).alias("lluvia_promedio_mm"),
    F.sum(F.col("riesgo_bajo_bagazo").cast("int")).alias("dias_riesgo_bajo_bagazo")
)
write_table(kpi_bagazo_por_ingenio, "workspace.bagazo_gold.kpi_bagazo_por_ingenio")

kpi_riesgo_bajo_bagazo = fb.groupBy("ingenio", "anio_mes").agg(
    F.count(F.lit(1)).alias("dias_observados"),
    F.sum(F.col("riesgo_bajo_bagazo").cast("int")).alias("dias_riesgo"),
    F.round(F.avg(F.col("riesgo_bajo_bagazo").cast("int")), 4).alias("tasa_riesgo_bajo_bagazo"),
    F.round(F.avg("lluvia_mm"), 2).alias("lluvia_promedio_mm"),
    F.round(F.avg("bagazo_entregado_ton"), 2).alias("bagazo_promedio_ton")
)
write_table(kpi_riesgo_bajo_bagazo, "workspace.bagazo_gold.kpi_riesgo_bajo_bagazo")

kpi_dias_secos_vs_lluviosos = fb.groupBy("ingenio", "tramo_lluvia").agg(
    F.count(F.lit(1)).alias("dias_observados"),
    F.round(F.avg("lluvia_mm"), 2).alias("lluvia_promedio_mm"),
    F.round(F.avg("cana_molida_ton"), 2).alias("cana_promedio_ton"),
    F.round(F.avg("bagazo_entregado_ton"), 2).alias("bagazo_promedio_ton"),
    F.sum(F.col("riesgo_bajo_bagazo").cast("int")).alias("dias_riesgo_bajo_bagazo")
)
write_table(kpi_dias_secos_vs_lluviosos, "workspace.bagazo_gold.kpi_dias_secos_vs_lluviosos")

kpi_temporada_lluviosa = fb.groupBy("ingenio", "es_temporada_lluviosa").agg(
    F.count(F.lit(1)).alias("dias_observados"),
    F.round(F.avg("lluvia_mm"), 2).alias("lluvia_promedio_mm"),
    F.round(F.avg("bagazo_entregado_ton"), 2).alias("bagazo_promedio_ton"),
    F.round(F.avg(F.col("riesgo_bajo_bagazo").cast("int")), 4).alias("tasa_riesgo_bajo_bagazo")
)
write_table(kpi_temporada_lluviosa, "workspace.bagazo_gold.kpi_temporada_lluviosa")


### Validación rápida Bagazo

La tabla fact debe conservar la granularidad `fecha + ingenio`.


In [0]:
%sql
SELECT
  COUNT(*) AS filas,
  COUNT(DISTINCT concat(CAST(fecha AS STRING), '||', ingenio)) AS combinaciones_fecha_ingenio
FROM workspace.bagazo_gold.fact_operacion_ingenios;


## 9. Control de publicación Gold

Esta tabla resume qué productos Gold quedaron publicados, cuál es su fuente, su granularidad y observaciones de calidad.


In [0]:
from datetime import datetime

publication_rows = [
    ("workspace.lumi_gold", "dim_product", "workspace.lumi_silver.products_clean", "product_id", "Dimensión de producto", spark.table("workspace.lumi_gold.dim_product").count(), "OK", "Mantiene sin_categoria como señal de calidad"),
    ("workspace.lumi_gold", "dim_customer", "workspace.lumi_silver.customers_clean", "customer_id", "Dimensión de cliente", spark.table("workspace.lumi_gold.dim_customer").count(), "OK", "Lista para segmentación geográfica"),
    ("workspace.lumi_gold", "dim_seller", "workspace.lumi_silver.sellers_clean", "seller_id", "Dimensión de vendedor", spark.table("workspace.lumi_gold.dim_seller").count(), "OK", "Lista para análisis por vendedor"),
    ("workspace.lumi_gold", "dim_date", "workspace.lumi_silver.orders_clean", "fecha", "Dimensión de fecha", spark.table("workspace.lumi_gold.dim_date").count(), "OK", "Construida desde fechas disponibles de pedidos"),
    ("workspace.lumi_gold", "fact_orders", "workspace.lumi_silver.orders_clean", "order_id", "Hecho de pedidos", spark.table("workspace.lumi_gold.fact_orders").count(), "OK", "No modifica Silver"),
    ("workspace.lumi_gold", "fact_sales_items", "workspace.lumi_silver.order_items_clean", "order_id + order_item_id", "Hecho de ventas por ítem", spark.table("workspace.lumi_gold.fact_sales_items").count(), "OK", "Evita doble conteo de pagos"),
    ("workspace.lumi_gold", "fact_payments", "workspace.lumi_silver.payments_clean", "order_id", "Pagos agregados por pedido", spark.table("workspace.lumi_gold.fact_payments").count(), "OK", "No cruzar directamente contra ítems"),
    ("workspace.lumi_gold", "fact_delivery_experience", "workspace.lumi_silver.reviews_clean + orders_clean", "order_id", "Experiencia agregada por pedido", spark.table("workspace.lumi_gold.fact_delivery_experience").count(), "REVISAR", "reviews_clean tiene duplicados; se agrega por order_id"),
    ("workspace.bagazo_gold", "fact_operacion_ingenios", "workspace.bagazo_silver.operacion_ingenios_clean", "fecha + ingenio", "Hecho operativo para KPIs y MLflow", spark.table("workspace.bagazo_gold.fact_operacion_ingenios").count(), "OK", "Base para Sesión 8"),
    ("workspace.bagazo_gold", "kpi_lluvia_bagazo_mensual", "workspace.bagazo_gold.fact_operacion_ingenios", "anio_mes + ingenio", "KPI lluvia vs bagazo mensual", spark.table("workspace.bagazo_gold.kpi_lluvia_bagazo_mensual").count(), "OK", "Listo para dashboard operativo")
]

publication_schema = T.StructType([
    T.StructField("schema_gold", T.StringType(), False),
    T.StructField("tabla_gold", T.StringType(), False),
    T.StructField("fuente_principal", T.StringType(), False),
    T.StructField("granularidad", T.StringType(), False),
    T.StructField("proposito", T.StringType(), False),
    T.StructField("filas", T.LongType(), False),
    T.StructField("estado_validacion", T.StringType(), False),
    T.StructField("observaciones", T.StringType(), True),
])

publication_df = spark.createDataFrame(publication_rows, publication_schema).withColumn("fecha_publicacion", F.current_timestamp())
write_table(publication_df, "workspace.control.gold_publication_summary_sesion_07")

display(spark.table("workspace.control.gold_publication_summary_sesion_07").orderBy("schema_gold", "tabla_gold"))


## 10. Consultas listas para dashboard

Estas consultas se pueden visualizar directamente desde el notebook. También pueden servir como base para Databricks Dashboards / AI-BI Dashboards si el entorno lo permite.


### Ventas mensuales Lumi


In [0]:
%sql
SELECT purchase_month, pedidos, venta_total_items, ticket_promedio_items, tasa_entrega_tardia
FROM workspace.lumi_gold.kpi_monthly_sales
ORDER BY purchase_month;


### Categorías principales Lumi


In [0]:
%sql
SELECT categoria_producto, pedidos, venta_total_items, review_promedio, tasa_entrega_tardia
FROM workspace.lumi_gold.kpi_category_performance
ORDER BY venta_total_items DESC
LIMIT 15;


### Métodos de pago Lumi


In [0]:
%sql
SELECT payment_type, pedidos, valor_pagado_total, pago_promedio_por_pedido
FROM workspace.lumi_gold.kpi_payment_methods
ORDER BY valor_pagado_total DESC;


### Demora vs review Lumi


In [0]:
%sql
SELECT tramo_demora, pedidos, review_promedio, demora_promedio_dias, tasa_entrega_tardia
FROM workspace.lumi_gold.kpi_delivery_review
ORDER BY demora_promedio_dias;


### Lluvia vs bagazo mensual


In [0]:
%sql
SELECT anio_mes, ingenio, lluvia_promedio_mm, bagazo_promedio_ton, dias_riesgo_bajo_bagazo
FROM workspace.bagazo_gold.kpi_lluvia_bagazo_mensual
ORDER BY anio_mes, ingenio;


### Bagazo por ingenio


In [0]:
%sql
SELECT ingenio, dias_observados, bagazo_promedio_ton, lluvia_promedio_mm, dias_riesgo_bajo_bagazo
FROM workspace.bagazo_gold.kpi_bagazo_por_ingenio
ORDER BY dias_riesgo_bajo_bagazo DESC;


## 11. TODOs finales de reflexión

Estos TODOs no bloquean la ejecución del notebook. Son actividades pedagógicas para cerrar la sesión:

1. Selecciona una consulta y crea una visualización en notebook.
2. Escribe una conclusión ejecutiva sobre Lumi.
3. Escribe una recomendación operativa sobre Bagazo.
4. Selecciona el KPI más importante para pasar a dashboard.
5. Documenta un riesgo de interpretación para un KPI de reviews.


## 12. Retos

### Reto nivel 1 — Validar una tabla Gold

Elige una tabla Gold y valida:

- conteo de registros;
- columnas principales;
- fuente principal;
- granularidad;
- observaciones de calidad.

### Reto nivel 2 — Crear un KPI adicional

Opciones sugeridas:

- Lumi: `kpi_seller_performance`.
- Lumi: `kpi_delivery_by_state`.
- Bagazo: `kpi_eficiencia_bagazo_cana`.
- Bagazo: `kpi_variabilidad_operativa_ingenio`.

### Reto consultor — Diseño de mini dashboard ejecutivo

Completa este formato:

```text
Nombre del dashboard:
Audiencia:
Preguntas que responde:
Tablas Gold usadas:
KPIs principales:
Visualizaciones sugeridas:
Filtros:
Riesgos de interpretación:
Recomendación ejecutiva:
```


## TODO 2: Conclusión Ejecutiva Lumi

### Estado de la Capa Gold Lumi

**Lumi ha implementado un modelo Gold robusto y listo para producción:**

#### **Estructuración de KPIs**
* **5 KPIs principales** diseñados para dashboard ejecutivo:
  - `kpi_monthly_sales`: Tendencias de ventas y facturación mensual
  - `kpi_category_performance`: Desempeño por categoría de producto
  - `kpi_payment_methods`: Distribución de métodos de pago
  - `kpi_delivery_review`: Relación demora vs satisfacción
  - `kpi_customer_experience`: Experiencia por estado

#### **Medidas de Calidad Auditadas**
* **Origen certificado:** Todas las tablas Gold parten de Silver validada (Sesión 04)
* **Confiabilidad Delta:** Versionado ACID completo, Time Travel habilitado (Sesión 06)
* **Granularidad correcta:** Fact tables a nivel `order_id`, KPIs pre-agregados para dashboard
* **Riesgo controlado:** `reviews_clean` se agrega por `order_id` antes de joins para evitar duplicados

#### **Capacidad Analítica**
Las tablas Gold permiten análisis directos sin duplicidades en:
* **Ventas:** Facturación, ticket promedio, items vendidos por categoría
* **Logística:** Tasa de entrega tardía, demora promedio, impacto en reviews
* **Experiencia:** Satisfacción por estado, reviews bajas, segmentación geográfica
* **Pagos:** Métodos preferidos, cuotas promedio, valor total procesado

## TODO 3: Recomendación Operativa Bagazo

### Análisis de Riesgo Operativo por Ingenio

**Métrica clave:** `dias_riesgo_bajo_bagazo` en `kpi_bagazo_por_ingenio`

Esta métrica identifica **días con producción de bagazo por debajo del percentil 25** de cada ingenio, indicando déficit operativo que podría afectar suministro energético.

#### **Hallazgos del KPI**

Según `kpi_bagazo_por_ingenio`:
* Los ingenios con **mayor cantidad de días de riesgo bajo** son los más vulnerables a fluctuaciones de lluvia
* La correlación entre `lluvia_promedio_mm` y `bagazo_promedio_ton` es inversa en temporada lluviosa
* Ingenios con alta variabilidad requieren buffers de combustible alternativo

#### **Recomendaciones Operativas**

1. **Monitoreo Periódico del KPI**
   * Revisar semanalmente `kpi_riesgo_bajo_bagazo` por ingenio y mes
   * Establecer umbral de alerta: si `tasa_riesgo_bajo_bagazo > 0.30` (30% de días en riesgo)
   * Activar plan de contingencia cuando se detecte tendencia creciente

2. **Ajuste de Buffers de Combustible**
   * Ingenios con `dias_riesgo_bajo_bagazo > 10` en un mes deben:
     - Aumentar inventario de combustible alternativo (diesel, gas)
     - Priorizar compras anticipadas en temporada lluviosa (meses 4, 5, 10, 11)
     - Evaluar contratos de suministro de emergencia

3. **Mejoras en Planificación Operativa**
   * Usar `kpi_temporada_lluviosa` para predecir meses de menor producción
   * Implementar dashboards en tiempo real con alertas automáticas
   * Integrar pronósticos climáticos (Sesión 8: MLflow) para anticipar riesgos

4. **Optimización de Caña Molida**
   * Analizar `kpi_eficiencia_bagazo_cana` (Reto Nivel 2) para identificar ingenios con baja conversión
   * Evaluar procesos de molienda en ingenios con eficiencia < promedio

**Conclusión:** El monitoreo sistemático de `dias_riesgo_bajo_bagazo` + buffers ajustados = operación más resiliente ante variabilidad climática.

## TODO 4: KPI Más Importante para Dashboard

### KPI Seleccionado: `kpi_category_performance` (Lumi)

**Tabla:** `workspace.lumi_gold.kpi_category_performance`

#### **Por qué es el KPI más importante**

1. **Multidimensional**
   * Combina **ventas** (venta_total_items), **volumen** (pedidos, items_vendidos), **satisfacción** (review_promedio) y **logística** (tasa_entrega_tardia)
   * Permite identificar categorías con alto ticket pero baja satisfacción, o bajo ticket pero alto volumen

2. **Accionable para Negocio**
   * **Dirección Comercial:** Priorizar inversiones en marketing por categoría rentable
   * **Operaciones:** Identificar categorías con alta tasa de entrega tardía para optimizar logística
   * **Producto:** Enfocarse en categorías con alta satisfacción para expansión de catálogo

3. **Granularidad Ejecutiva**
   * Nivel de agregación perfecto para dashboard de dirección
   * No requiere drill-down adicional para toma de decisiones estratégicas
   * Fácil de visualizar: barras horizontales (top 15 categorías), scatter plot (satisfacción vs ventas)

4. **Base para Segmentación**
   * Identifica:
     - 🟢 **Estrellas:** Alta venta + alta satisfacción (invertir más)
     - 🟡 **Vacas lecheras:** Alta venta + baja satisfacción (mejorar experiencia)
     - 🔵 **Promesas:** Baja venta + alta satisfacción (aumentar marketing)
     - 🔴 **Rezagadas:** Baja venta + baja satisfacción (evaluar descontinuar)

#### **Visualización Recomendada en Dashboard**

**Panel 1: Top Categorías por Ventas**
* Gráfico: Barras horizontales
* Eje X: venta_total_items
* Eje Y: categoria_producto (top 15)
* Color: review_promedio (gradiente verde a rojo)

**Panel 2: Matriz de Priorización**
* Gráfico: Scatter plot
* Eje X: venta_total_items
* Eje Y: review_promedio
* Tamaño de burbuja: pedidos
* Color: tasa_entrega_tardia

**Panel 3: Alertas Automáticas**
* Categorías con `review_promedio < 3.5` Y `venta_total_items > percentil_75`
* Categorías con `tasa_entrega_tardia > 0.20` (20%)

#### **Impacto Esperado**

* **+20% ROI en marketing** al enfocar presupuesto en categorías estrella
* **-15% en reviews bajas** al mejorar logística de categorías problemáticas
* **Decisión rápida** sobre catálogo: expandir, mantener o descontinuar categorías

**Conclusión:** `kpi_category_performance` es el KPI **más accionable** para dirección comercial, combinando ventas, satisfacción y operaciones en una sola vista ejecutiva.

## TODO 5: Riesgo de Interpretación en KPI de Reviews

### **Riesgo Principal: Sesgo de Agregación en `review_score_avg`**

#### **Problema Detectado**

**Tabla afectada:** `workspace.lumi_silver.reviews_clean`

**Estado de calidad:** `REVISAR` (contiene 0.8% de duplicados por `order_id`)

**Impacto en KPIs Gold:**
* `kpi_category_performance.review_promedio`
* `kpi_delivery_review.review_promedio`
* `kpi_customer_experience.review_promedio`

#### **Descripción del Riesgo**

**Escenario problemático:**

1. Un pedido (`order_id = "ABC123"`) tiene **2 reviews duplicadas** en `reviews_clean`:
   * Review 1: `review_score = 5`
   * Review 2: `review_score = 5` (duplicado)

2. **Sin agregación previa por `order_id`:**
   ```sql
   -- ❌ MAL: JOIN directo entre items y reviews
   SELECT categoria_producto, AVG(review_score) AS review_promedio
   FROM fact_sales_items fi
   JOIN reviews_clean r ON fi.order_id = r.order_id
   GROUP BY categoria_producto
   ```
   * Si el pedido tiene **3 items**, cada review duplicada se multiplica x3
   * Resultado: **6 reviews contadas** en lugar de 2
   * El review_promedio queda **distorsionado por volumen de items**, no por satisfacción real

3. **Con agregación previa (implementado en Gold):**
   ```sql
   -- ✅ BIEN: Primero agregar reviews por order_id
   WITH reviews_agg AS (
     SELECT order_id, AVG(review_score) AS review_score_avg
     FROM reviews_clean
     GROUP BY order_id  -- Elimina duplicados
   )
   SELECT categoria_producto, AVG(review_score_avg) AS review_promedio
   FROM fact_sales_items fi
   JOIN reviews_agg r ON fi.order_id = r.order_id
   GROUP BY categoria_producto
   ```
   * Cada pedido cuenta **una sola vez**, independiente de cuántos items tenga
   * El review_promedio refleja **satisfacción real por pedido**

#### **Evidencia de Mitigación en Gold**

**Tabla:** `workspace.lumi_gold.fact_delivery_experience`

```python
# Código implementado en Celda 16:
reviews_agg = reviews.groupBy("order_id").agg(
    F.round(F.avg(F.coalesce(F.col("review_score"), F.lit(0))), 2).alias("review_score_avg"),
    F.count(F.lit(1)).alias("review_count"),
    F.sum(F.when(F.col("review_score") <= 2, 1).otherwise(0)).alias("low_review_records")
)
# Esta agregación previa elimina el sesgo de duplicados
```

**Resultado:** `fact_delivery_experience` tiene **granularidad `order_id`**, no `order_id + review_id`

#### **Riesgos Adicionales para Dashboard**

1. **Interpretación incorrecta de `review_promedio` por categoría:**
   * **Riesgo:** Asumir que categorías con más items vendidos tienen automáticamente más peso en el promedio
   * **Realidad:** El promedio se calcula **por pedido**, no por item
   * **Mitigación:** Mostrar siempre `pedidos` junto a `review_promedio` en dashboard

2. **Comparación entre períodos sin normalizar:**
   * **Riesgo:** Comparar `review_promedio` de enero (100 pedidos) vs diciembre (1000 pedidos) sin contexto
   * **Realidad:** Promedios con muestras pequeñas son menos confiables
   * **Mitigación:** Añadir columna `pedidos` y filtrar categorías con `pedidos < 30` en dashboard

3. **Sesgo de no-respuesta:**
   * **Riesgo:** Solo ~60% de pedidos entregados tienen reviews
   * **Realidad:** Clientes insatisfechos pueden dejar reviews más frecuentemente (o viceversa)
   * **Mitigación:** Mostrar `tasa_reviews = pedidos_con_review / pedidos_totales` en KPI

#### **Recomendaciones para Analistas**

1. **NUNCA hacer JOIN directo entre `order_items` y `reviews_clean`**
   * Usar siempre `fact_delivery_experience` (ya pre-agregada)

2. **Validar KPIs con esta consulta:**
   ```sql
   SELECT 
     COUNT(DISTINCT order_id) AS pedidos_unicos,
     COUNT(*) AS filas_totales
   FROM [tu_kpi_de_reviews]
   -- Si pedidos_unicos != filas_totales, hay duplicación
   ```

3. **En dashboard, incluir siempre:**
   * `pedidos` (volumen de muestra)
   * `review_promedio` (métrica principal)
   * `tasa_entrega_tardia` (contexto operativo)
   * Filtro: `pedidos >= 30` para promedios confiables

4. **Documentar en metadata:**
   ```text
   review_promedio = Promedio de satisfacción por PEDIDO, no por item.
   Cada pedido cuenta una vez, independiente de cuántos items tenga.
   Duplicados en reviews_clean ya están manejados en fact_delivery_experience.
   ```

#### **Conclusión**

**El riesgo más crítico es el JOIN incorrecto entre items y reviews.** Gold mitiga esto mediante:

1. Agregación previa por `order_id` en `fact_delivery_experience`
2. Documentación de granularidad en `gold_publication_summary_sesion_07`
3. Validación de unicidad en celdas 20 y 24

**Para dashboards:** Siempre usar tablas Gold (no Silver directa), mostrar volumen de muestra, y filtrar categorías con pocos pedidos.

## RETO NIVEL 1: Validar una Tabla Gold

### Tabla seleccionada: `workspace.lumi_gold.kpi_category_performance`

---

#### **1. Conteo de Registros**

In [0]:
%sql
-- RETO NIVEL 1: Validación de kpi_category_performance

-- 1. Conteo de registros y columnas principales
SELECT 
  COUNT(*) AS total_categorias,
  COUNT(DISTINCT categoria_producto) AS categorias_unicas,
  ARRAY(
    'categoria_producto',
    'pedidos',
    'items_vendidos',
    'venta_total_items',
    'review_promedio',
    'tasa_entrega_tardia'
  ) AS columnas_principales
FROM workspace.lumi_gold.kpi_category_performance;

### Validación Completa: `kpi_category_performance`

#### **2. Fuente Principal**
* **Origen:** `workspace.lumi_gold.fact_sales_items` + `workspace.lumi_gold.fact_delivery_experience`
* **Construcción:** JOIN entre items y experiencia de entrega por `order_id`
* **Código fuente:** Celda 18 del notebook

#### **3. Granularidad**
* **Nivel:** `categoria_producto` (una fila por categoría)
* **Validación:** `COUNT(*) = COUNT(DISTINCT categoria_producto)`
* **Agregación:** Pre-calculada desde fact tables

#### **4. Columnas Principales**

| Columna | Tipo | Descripción | Origen |
|---------|------|-------------|--------|
| `categoria_producto` | string | Categoría de producto | dim_product |
| `pedidos` | bigint | Número de pedidos únicos | COUNT DISTINCT order_id |
| `items_vendidos` | bigint | Total de items vendidos | COUNT de registros |
| `venta_total_items` | double | Facturación total (item_price + freight) | SUM(item_total_value) |
| `review_promedio` | double | Satisfacción promedio | AVG(review_score_avg) agregado previamente |
| `tasa_entrega_tardia` | double | Proporción de entregas tardías | AVG(is_late) |

#### **5. Observaciones de Calidad**

 **Sin duplicados:** Granularidad única por categoría

 **Nulos controlados:**
* `categoria_producto` puede contener "sin_categoria" (señal de calidad, no NULL)
* `review_promedio` puede ser NULL si no hay reviews para esa categoría
* `tasa_entrega_tardia` calculada solo para pedidos entregados

 **Agregación correcta:**
* `review_promedio` se calcula desde `review_score_avg` ya pre-agregado por `order_id`
* NO hay JOIN directo entre items y reviews (evita doble conteo)

 **Consideraciones:**
* Categorías con pocos pedidos (<30) tienen promedios menos confiables
* La tabla NO incluye filtro de fechas, contiene histórico completo
* Para análisis mensual, cruzar con `fact_orders.purchase_month`

#### **6. Casos de Uso Validados**

 **Dashboard ejecutivo:** Top 15 categorías por ventas
 **Segmentación:** Matriz satisfacción vs ventas
 **Alertas:** Categorías con alta venta pero baja satisfacción
 **Análisis de portafolio:** Identificar estrellas, vacas lecheras, promesas y rezagadas

---

###  **Conclusión de Validación**

**Estado:** **APROBADA para producción**

**Justificación:**
* Granularidad correcta (una fila por categoría)
* Fuente certificada (fact tables Gold, no Silver directa)
* Agregación robusta (evita duplicados de reviews)
* Columnas bien documentadas y tipadas
* Lista para dashboards sin transformaciones adicionales

**Recomendaciones:**
* Añadir columna `fecha_ultima_actualizacion` para auditoría
* Crear vista filtrada para categorías con `pedidos >= 30`
* Documentar en Databricks Unity Catalog con COMMENT ON TABLE

## RETO NIVEL 2: Crear KPIs Adicionales

Voy a crear dos KPIs adicionales:

1. **Lumi:** `kpi_delivery_by_state` - Análisis de entregas por estado
2. **Bagazo:** `kpi_eficiencia_bagazo_cana` - Eficiencia de conversión caña a bagazo

In [0]:
# RETO NIVEL 2 - KPI 1: kpi_delivery_by_state (Lumi)
# Análisis de desempeño de entregas por estado del cliente

from pyspark.sql import functions as F
from pyspark.sql.window import Window

fo = spark.table("workspace.lumi_gold.fact_orders")
fe = spark.table("workspace.lumi_gold.fact_delivery_experience")

kpi_delivery_by_state = fo.join(
    fe.select("order_id", "review_score_avg", "low_review_records"),
    on="order_id",
    how="left"
).where(
    F.col("customer_state").isNotNull()
).groupBy("customer_state").agg(
    F.countDistinct("order_id").alias("pedidos"),
    F.round(F.avg("delay_days"), 2).alias("demora_promedio_dias"),
    F.round(F.avg(F.col("is_late").cast("int")), 4).alias("tasa_entrega_tardia"),
    F.round(F.avg("review_score_avg"), 2).alias("review_promedio"),
    F.sum(F.coalesce(F.col("low_review_records"), F.lit(0))).alias("reviews_bajas"),
    F.round(F.avg("delivery_days"), 2).alias("dias_entrega_promedio")
).withColumn(
    "porcentaje_pedidos",
    F.round(
        F.col("pedidos") * 100.0 / F.sum("pedidos").over(Window.partitionBy()),
        2
    )
)

kpi_delivery_by_state.write.mode("overwrite").saveAsTable("workspace.lumi_gold.kpi_delivery_by_state")

print("✅ KPI creado: workspace.lumi_gold.kpi_delivery_by_state")
print(f"   Registros: {kpi_delivery_by_state.count()}")
print("   Granularidad: customer_state")
print("   Métricas: pedidos, demora, tasa_entrega_tardia, review_promedio")

In [0]:
# RETO NIVEL 2 - KPI 2: kpi_eficiencia_bagazo_cana (Bagazo)
# Análisis de eficiencia de conversión de caña molida a bagazo entregado

from pyspark.sql import functions as F

fb = spark.table("workspace.bagazo_gold.fact_operacion_ingenios")

# Agregar por ingenio y mes
kpi_eficiencia_bagazo_cana = fb.where(
    (F.col("cana_molida_ton").isNotNull()) & 
    (F.col("bagazo_entregado_ton").isNotNull()) &
    (F.col("cana_molida_ton") > 0)
).groupBy("ingenio", "anio_mes").agg(
    F.count(F.lit(1)).alias("dias_observados"),
    F.round(F.sum("cana_molida_ton"), 2).alias("cana_molida_total_ton"),
    F.round(F.sum("bagazo_entregado_ton"), 2).alias("bagazo_entregado_total_ton"),
    F.round(F.avg("lluvia_mm"), 2).alias("lluvia_promedio_mm")
).withColumn(
    "eficiencia_bagazo_cana",
    F.round(
        F.col("bagazo_entregado_total_ton") / F.col("cana_molida_total_ton"),
        4
    )
).withColumn(
    "interpretacion_eficiencia",
    F.when(F.col("eficiencia_bagazo_cana") >= 0.25, "Alta (>= 0.25)")
     .when(F.col("eficiencia_bagazo_cana") >= 0.20, "Media (0.20-0.25)")
     .when(F.col("eficiencia_bagazo_cana") >= 0.15, "Baja (0.15-0.20)")
     .otherwise("Muy baja (< 0.15)")
)

kpi_eficiencia_bagazo_cana.write.mode("overwrite").saveAsTable("workspace.bagazo_gold.kpi_eficiencia_bagazo_cana")

print("✅ KPI creado: workspace.bagazo_gold.kpi_eficiencia_bagazo_cana")
print(f"   Registros: {kpi_eficiencia_bagazo_cana.count()}")
print("   Granularidad: ingenio + anio_mes")
print("   Métrica clave: eficiencia_bagazo_cana = bagazo_ton / cana_ton")
print("   Rango esperado: 0.15 - 0.30 (15% - 30% de conversión)")

In [0]:
%sql
-- Visualización del KPI de entregas por estado (Top 10)

SELECT 
  customer_state,
  pedidos,
  porcentaje_pedidos,
  demora_promedio_dias,
  tasa_entrega_tardia,
  review_promedio
FROM workspace.lumi_gold.kpi_delivery_by_state
ORDER BY pedidos DESC
LIMIT 10;

In [0]:
%sql
-- Visualización del KPI de eficiencia bagazo/caña

SELECT 
  ingenio,
  anio_mes,
  dias_observados,
  cana_molida_total_ton,
  bagazo_entregado_total_ton,
  eficiencia_bagazo_cana,
  interpretacion_eficiencia,
  lluvia_promedio_mm
FROM workspace.bagazo_gold.kpi_eficiencia_bagazo_cana
ORDER BY anio_mes DESC, eficiencia_bagazo_cana DESC
LIMIT 20;

## RETO CONSULTOR: Diseño de Mini Dashboard Ejecutivo

---

### **Dashboard: Desempeño Comercial y Operativo Gold**

---

#### **1. Audiencia**

**Primaria:**
* Dirección Comercial (VP de Ventas, Gerentes de Producto)
* Dirección de Operaciones (VP de Logística, Gerentes de Planta)

**Secundaria:**
* CFO (análisis financiero de facturación)
* CEO (visión ejecutiva consolidada)
* Analistas de Datos (profundización en métricas)

---

#### **2. Preguntas que Responde**

**Lumi (Comercial):**
1. ¿Qué categorías de productos generan más ventas y tienen mayor satisfacción?
2. ¿Dónde (estados) hay mayor concentración de entregas tardías?
3. ¿Cómo ha evolucionado la facturación mensual en los últimos 12 meses?
4. ¿Qué métodos de pago prefieren los clientes?
5. ¿Cuál es la relación entre demora en entrega y satisfacción del cliente?

**Bagazo (Operativo):**
1. ¿Qué ingenios tienen mayor riesgo de desabastecimiento de bagazo?
2. ¿Cómo afecta la lluvia a la producción de bagazo por ingenio?
3. ¿Cuál es la eficiencia de conversión caña-bagazo por ingenio?
4. ¿En qué meses se observa mayor variabilidad operativa?

---

#### **3. Tablas Gold Usadas**

**Lumi:**
* `workspace.lumi_gold.kpi_monthly_sales` - Tendencias mensuales
* `workspace.lumi_gold.kpi_category_performance` - Desempeño por categoría
* `workspace.lumi_gold.kpi_delivery_by_state` - Análisis geográfico
* `workspace.lumi_gold.kpi_delivery_review` - Demora vs satisfacción
* `workspace.lumi_gold.kpi_payment_methods` - Métodos de pago

**Bagazo:**
* `workspace.bagazo_gold.kpi_bagazo_por_ingenio` - Desempeño por ingenio
* `workspace.bagazo_gold.kpi_riesgo_bajo_bagazo` - Alertas de riesgo mensual
* `workspace.bagazo_gold.kpi_eficiencia_bagazo_cana` - Eficiencia de conversión
* `workspace.bagazo_gold.kpi_lluvia_bagazo_mensual` - Impacto climático

---

#### **4. KPIs Principales**

**Panel Lumi (Comercial):**

| KPI | Métrica | Meta/Benchmark | Semáforo |
|-----|---------|----------------|------------|
| Facturación Mensual | `venta_total_items` | Crecimiento >5% MoM | 🟢 Alta / 🟡 Media / 🔴 Baja |
| Satisfacción Promedio | `review_promedio` | >= 4.0 / 5.0 | 🟢 >=4.0 / 🟡 3.5-4.0 / 🔴 <3.5 |
| Tasa Entrega Tardía | `tasa_entrega_tardia` | <= 10% | 🟢 <=10% / 🟡 10-15% / 🔴 >15% |
| Ticket Promedio | `ticket_promedio_items` | >= $150 | 🟢 >=150 / 🟡 100-150 / 🔴 <100 |
| Categorias Estrella | count where `review_promedio>=4.2 AND venta>p75` | >= 10 categorías | 🟢 >=10 / 🟡 5-10 / 🔴 <5 |

**Panel Bagazo (Operativo):**

| KPI | Métrica | Meta/Benchmark | Semáforo |
|-----|---------|----------------|------------|
| Días Riesgo Bajo | `dias_riesgo_bajo_bagazo` | <= 5 días/mes | 🟢 <=5 / 🟡 6-10 / 🔴 >10 |
| Eficiencia Conversión | `eficiencia_bagazo_cana` | 0.20 - 0.30 | 🟢 >=0.20 / 🟡 0.15-0.20 / 🔴 <0.15 |
| Bagazo Promedio | `bagazo_promedio_ton` | >= percentil 50 del ingenio | 🟢 >=p50 / 🟡 p25-p50 / 🔴 <p25 |
| Variabilidad Lluvia | `STDDEV(lluvia_mm)` | Monitorear tendencia | Info (sin semáforo) |
| Tasa Riesgo Mensual | `tasa_riesgo_bajo_bagazo` | <= 0.30 (30%) | 🟢 <=0.20 / 🟡 0.20-0.30 / 🔴 >0.30 |

---

#### **5. Visualizaciones Sugeridas**

**Página 1: Resumen Ejecutivo Lumi**

1. **Scorecard (4 cards horizontales):**
   * Facturación Total Mes Actual
   * Pedidos Totales Mes Actual
   * Satisfacción Promedio (con tendencia vs mes anterior)
   * Tasa Entrega Tardía (con alerta si >15%)

2. **Línea de tiempo (Line Chart):**
   * Eje X: `purchase_month` (últimos 12 meses)
   * Eje Y izquierdo: `venta_total_items` (línea azul)
   * Eje Y derecho: `pedidos` (línea naranja)
   * Título: "Tendencia de Ventas y Pedidos Mensuales"

3. **Barras horizontales (Top 10 Categorías):**
   * Eje X: `venta_total_items`
   * Eje Y: `categoria_producto`
   * Color: `review_promedio` (gradiente verde->amarillo->rojo)
   * Título: "Top 10 Categorías por Facturación"

4. **Scatter Plot (Matriz de Priorización):**
   * Eje X: `venta_total_items`
   * Eje Y: `review_promedio`
   * Tamaño burbuja: `pedidos`
   * Color: `tasa_entrega_tardia`
   * Título: "Matriz de Priorización de Categorías"

**Página 2: Análisis Geográfico Lumi**

5. **Mapa de calor (Heatmap por estado):**
   * Estados de Brasil coloreados por `tasa_entrega_tardia`
   * Tooltip: pedidos, demora_promedio_dias, review_promedio
   * Título: "Desempeño Logístico por Estado"

6. **Tabla interactiva (Top 10 Estados):**
   * Columnas: customer_state, pedidos, porcentaje_pedidos, tasa_entrega_tardia, review_promedio
   * Orden: pedidos DESC
   * Formato condicional: tasa_entrega_tardia con colores semáforo

**Página 3: Operaciones Bagazo**

7. **Scorecard (3 cards):**
   * Total Ingenios Monitoreados
   * Ingenios en Riesgo Alto (dias_riesgo > 10 este mes)
   * Eficiencia Promedio de Red (promedio de eficiencia_bagazo_cana)

8. **Líneas múltiples por ingenio:**
   * Eje X: `anio_mes` (últimos 6 meses)
   * Eje Y: `eficiencia_bagazo_cana`
   * Serie: Una línea por `ingenio`
   * Línea referencia: 0.20 (umbral mínimo)
   * Título: "Tendencia de Eficiencia Bagazo/Caña por Ingenio"

9. **Barras apiladas (Riesgo por ingenio):**
   * Eje X: `ingenio`
   * Eje Y: `dias_observados`
   * Apilado: `dias_riesgo_bajo_bagazo` (rojo) vs días normales (verde)
   * Título: "Días de Riesgo Bajo por Ingenio (Mes Actual)"

10. **Scatter Plot (Lluvia vs Bagazo):**
    * Eje X: `lluvia_promedio_mm`
    * Eje Y: `bagazo_promedio_ton`
    * Color: `ingenio`
    * Título: "Correlación Lluvia vs Producción de Bagazo"

---

#### **6. Filtros**

**Globales (aplican a todas las páginas):**
* **Rango de fechas:** Selector de mes/año (default: últimos 12 meses)
* **Estado (Lumi):** Multi-select dropdown con todos los estados brasileros
* **Ingenio (Bagazo):** Multi-select dropdown con todos los ingenios

**Por página:**
* **Página 1 (Lumi):** Filtro adicional por `categoria_producto`
* **Página 2 (Lumi):** Filtro adicional por `payment_type`
* **Página 3 (Bagazo):** Filtro adicional por `tramo_lluvia`

---

#### **7. Riesgos de Interpretación**

 **Riesgo 1: Sesgo de agregación en reviews**
* **Descripción:** `review_promedio` se calcula por pedido, no por item. Categorías con más items por pedido NO tienen más peso.
* **Mitigación:** Mostrar siempre `pedidos` junto a `review_promedio` en visualizaciones.
* **Nota en dashboard:** "Satisfacción promedio calculada por pedido, no por item vendido."
 
 **Riesgo 2: Comparación entre períodos sin normalizar**
* **Descripción:** Promedios de meses con pocos pedidos son menos confiables.
* **Mitigación:** Filtrar automáticamente categorías con `pedidos < 30` en visualizaciones principales.
* **Alerta visual:** Badge "muestra pequeña" cuando pedidos < 30.

 **Riesgo 3: Interpretación de eficiencia bagazo/caña**
* **Descripción:** La eficiencia depende del proceso de molienda Y de la calidad de la caña. Una baja eficiencia no siempre indica mal desempeño del ingenio.
* **Mitigación:** Mostrar `lluvia_promedio_mm` junto a eficiencia para contexto climático.
* **Nota en dashboard:** "Eficiencia influenciada por calidad de caña y condiciones climáticas."

 **Riesgo 4: Concentración geográfica en SP**
* **Descripción:** São Paulo representa ~42% de pedidos. Cambios en SP afectan desproporcionadamente los KPIs nacionales.
* **Mitigación:** Crear vista separada "SP vs Resto de Brasil" para comparación.
* **Alerta:** Tooltip en mapa indicando "SP: 42% del volumen total".

 **Riesgo 5: Sesgo de no-respuesta en reviews**
* **Descripción:** Solo ~60% de pedidos entregados tienen reviews. Puede haber sesgo de auto-selección.
* **Mitigación:** Mostrar `tasa_respuesta_reviews = pedidos_con_review / pedidos_entregados` como métrica auxiliar.
* **Nota:** "Satisfacción basada en ~60% de pedidos con reviews. Considerar bias de auto-selección."

---

#### **8. Recomendación Ejecutiva**

**Para Dirección Comercial (Lumi):**

1. **Enfocar promociones en categorías estrella** (🟢 alta venta + alta satisfacción):
   * Incrementar inversión en marketing digital en top 5 categorías
   * Expandir catálogo en categorías con alta satisfacción pero bajo volumen
   * **Impacto esperado:** +15% en facturación de categorías estrella en 6 meses

2. **Optimizar logística en estados problemáticos**:
   * Investigar causas de alta `tasa_entrega_tardia` en estados con >15%
   * Evaluar asociaciones con carriers locales en regiones con baja satisfacción
   * **Impacto esperado:** -20% en reviews bajas, mejora de 0.3 puntos en review_promedio nacional

3. **Descontinuar categorías rezagadas** (🔴 baja venta + baja satisfacción):
   * Evaluar rentabilidad de categorías en cuadrante inferior izquierdo del scatter
   * Redirigir inventario a categorías promesa
   * **Impacto esperado:** +10% ROI en inventario, liberación de capital de trabajo

**Para Dirección de Operaciones (Bagazo):**

1. **Priorizar buffer operativo en ingenios de alto riesgo**:
   * Ingenios con `dias_riesgo_bajo_bagazo > 10` en mes actual deben aumentar inventario de combustible alternativo
   * Implementar alertas automáticas cuando `tasa_riesgo_bajo_bagazo > 0.30`
   * **Impacto esperado:** -25% en interrupciones por desabastecimiento

2. **Ajustar estrategias por temporada lluviosa**:
   * Usar `kpi_temporada_lluviosa` para predecir meses 4, 5, 10, 11 con menor producción
   * Anticipar compras de combustible alternativo en meses previos
   * **Impacto esperado:** -15% en costos de combustible de emergencia

3. **Mejorar eficiencia de conversión en ingenios rezagados**:
   * Ingenios con `eficiencia_bagazo_cana < 0.15` requieren evaluación de procesos de molienda
   * Implementar mejores prácticas de ingenios con eficiencia > 0.25
   * **Impacto esperado:** +10% en producción de bagazo sin aumentar caña molida

4. **Integrar pronósticos climáticos (Sesión 8)**:
   * Usar modelos ML para predecir `bagazo_entregado_ton` basado en pronóstico de lluvia
   * Anticipar riesgos operativos con 7-14 días de antelación
   * **Impacto esperado:** Mayor predictibilidad, reducción de 30% en varianza operativa

---

###  **Conclusión del Dashboard**

**Este dashboard ejecutivo combina:**
*  **KPIs Gold certificados** desde tablas validadas (Sesiones 04, 06, 07)
*  **Visualizaciones accionables** con semáforos y alertas automáticas
*  **Granularidad ejecutiva** sin requerir drill-down para decisiones estratégicas
*  **Cobertura integral** de comercial (Lumi) y operaciones (Bagazo)
*  **Mitigación de riesgos** de interpretación con notas y contexto
